# KA-1 adaptation and low-rank arithmetic
Prerequisite: Primer-PY and the declared CPU environment. Reuses the existing Mini GPT and KA-1 files; it re-generates all 144 attempts without overwriting the retained experiment.

All textbook conclusions are also visible in Chapters 34–39. Original narrative/data: CC BY-SA 4.0. Code: Apache-2.0. No model training occurs.


In [1]:
from pathlib import Path
import sys
ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src/config/book.mjs").exists())
sys.path.insert(0, str(ROOT / "code/part-vi"))
from core import *
from run import run, verify
record = verify(run())
print("Hand-derived expectations verified; model training: none")

Hand-derived expectations verified; model training: none


In [2]:
import json
sys.path.insert(0, str(ROOT / "code/knowledge-assistant"))
from format_adaptation import run as run_adaptation, key
actual = run_adaptation()
saved = json.loads((ROOT / "data/knowledge-assistant/ka1-format-run-v1.json").read_text())
prior = {key(row): row for row in saved["rows"]}
for row in actual["rows"]:
    for field in ("output", "normalized", "raw_strict", "normalized_strict", "semantic_score"):
        assert row[field] == prior[key(row)][field]
assert actual["parameters_unchanged"]
print("Exact generated outputs reproduced:", len(actual["rows"]), "; parameter updates: none")

Exact generated outputs reproduced: 144 ; parameter updates: none


In [3]:
for summary in actual["summary"]:
    if summary["policy"] == "greedy":
        print(summary["candidate"], summary["locale"], summary["arm"],
              "strict", summary["strict_passes"], "/18; semantic", summary["semantic_passes"],
              "/18; stable", summary["stable_semantic_cases"], "/6;",
              summary["decision"])
print("Semantic labels are the disclosed original author judgments, reused only after exact raw replay.")

final-600 en raw strict 0 /18; semantic 0 /18; stable 0 /6; reject
final-600 en normalized strict 0 /18; semantic 0 /18; stable 0 /6; reject
final-600 zh-hans raw strict 0 /18; semantic 0 /18; stable 0 /6; reject
final-600 zh-hans normalized strict 0 /18; semantic 0 /18; stable 0 /6; reject
selected-50 en raw strict 0 /18; semantic 6 /18; stable 2 /6; reject
selected-50 en normalized strict 6 /18; semantic 6 /18; stable 2 /6; reject
selected-50 zh-hans raw strict 0 /18; semantic 0 /18; stable 0 /6; reject
selected-50 zh-hans normalized strict 0 /18; semantic 0 /18; stable 0 /6; reject
Semantic labels are the disclosed original author judgments, reused only after exact raw replay.


In [4]:
data = fixture("low-rank-v1.json")
result = low_rank(data["W0"], data["A"], data["B"], data["x"])
print("Assigned matrices, not trained weights:", result)
assert result["combined_output"] == [0, 2, 4, 1] == result["merged_output"]
assert low_rank(data["W0"], data["A"], data["B"], [1, 0, 0])["combined_output"] == [3, 0, -2, 5]
print("Large-layer trainable fraction:", 8 * (4096 + 4096) / 4096**2)
print("Quantization illustration, not NF4:", record["quantization"])
print("No SFT, LoRA or QLoRA training benchmark has been run.")

Assigned matrices, not trained weights: {'delta': [[2.0, 0.0, -1.0], [0.0, 0.0, 0.0], [-2.0, 0.0, 1.0], [4.0, 0.0, -2.0]], 'base_output': [1, 2, 3, 3], 'adapter_output': [-1.0, 0.0, 1.0, -2.0], 'combined_output': [0.0, 2.0, 4.0, 1.0], 'merged_output': [0.0, 2.0, 4.0, 1.0], 'base_parameters': 12, 'trainable_parameters': 7}
Large-layer trainable fraction: 0.00390625
Quantization illustration, not NF4: {'original': 1.86, 'quantized': 2.0, 'original_with_delta': 1.9600000000000002, 'quantized_with_delta': 2.1}
No SFT, LoRA or QLoRA training benchmark has been run.
